# ***Model Training & Baseline Comparison***

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import BallTree
import joblib

In [2]:
import gdown

FILE_ID = "1TlXTBX9f97idNgPneY0mMsLrzqmWYrOy"
OUTPUT  = "features_labels_events.csv"

# Download file dari Google Drive
url = f"https://drive.google.com/uc?id={FILE_ID}"
gdown.download(url, OUTPUT, quiet=False)


Downloading...
From (original): https://drive.google.com/uc?id=1TlXTBX9f97idNgPneY0mMsLrzqmWYrOy
From (redirected): https://drive.google.com/uc?id=1TlXTBX9f97idNgPneY0mMsLrzqmWYrOy&confirm=t&uuid=78b9d76c-c6c0-4afb-a91f-c6b20b28aaf3
To: /content/features_labels_events.csv
100%|██████████| 444M/444M [00:01<00:00, 263MB/s]


'features_labels_events.csv'

In [3]:
df = pd.read_csv('features_labels_events.csv', parse_dates=['Date'])

FEATURE_COLUMNS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    'is_weekend', 'grid_crime_count', 'grid_arrest_rate',
    'grid_theft_share', 'grid_narcotics_share', 'Latitude', 'Longitude',
]
TARGET_COLUMN = 'risk_score'

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data latih: {X_train.shape[0]} baris")
print(f"Data uji  : {X_test.shape[0]} baris")

Data latih: 999912 baris
Data uji  : 249979 baris


In [5]:
baseline_pred = np.full(len(y_test), y_train.mean())

In [6]:
model = RandomForestRegressor(
    n_estimators=50,
    max_depth=12,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

In [7]:
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{label:10s} | MAE={mae:6.3f} | RMSE={rmse:6.3f} | R2={r2:6.3f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}


print('Perbandingan Baseline vs Model')
print('-' * 55)
baseline_metrics = evaluate(y_test, baseline_pred, 'Baseline')
model_metrics = evaluate(y_test, model_pred, 'Model')

improvement_mae = (baseline_metrics['mae'] - model_metrics['mae']) / baseline_metrics['mae'] * 100
print(f"\nPerbaikan MAE terhadap baseline: {improvement_mae:.2f} persen")

Perbandingan Baseline vs Model
-------------------------------------------------------
Baseline   | MAE= 9.546 | RMSE=12.391 | R2=-0.000
Model      | MAE= 3.082 | RMSE= 4.338 | R2= 0.877

Perbaikan MAE terhadap baseline: 67.71 persen


In [8]:
import gdown

FILE_ID = "1MOmSeDU9uO8mRd6gL12n64MTIAQsBSaO"
OUTPUT  = "risk_lookup_table.csv"

# Download file dari Google Drive
url = f"https://drive.google.com/uc?id={FILE_ID}"
gdown.download(url, OUTPUT, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1MOmSeDU9uO8mRd6gL12n64MTIAQsBSaO
To: /content/risk_lookup_table.csv
100%|██████████| 8.12M/8.12M [00:00<00:00, 67.4MB/s]


'risk_lookup_table.csv'

In [9]:
lookup_grids = pd.read_csv('risk_lookup_table.csv')[
    ['grid_id', 'grid_lat', 'grid_lon']
].drop_duplicates('grid_id')

grid_feature_ref = (
    df.groupby('grid_id')
    .agg(
        grid_crime_count=('grid_crime_count', 'first'),
        grid_arrest_rate=('grid_arrest_rate', 'first'),
        grid_theft_share=('grid_theft_share', 'first'),
        grid_narcotics_share=('grid_narcotics_share', 'first'),
    )
    .reset_index()
    .merge(lookup_grids, on='grid_id', how='left')
)

print("Jumlah grid unik dengan fitur historis:", len(grid_feature_ref))

Jumlah grid unik dengan fitur historis: 733


In [10]:
os.makedirs('outputs', exist_ok=True)
joblib.dump(model, 'outputs/risk_model_v1.joblib')
grid_feature_ref.to_csv('outputs/grid_feature_reference.csv', index=False)
print("Tersimpan: outputs/risk_model_v1.joblib")
print("Tersimpan: outputs/grid_feature_reference.csv")

Tersimpan: outputs/risk_model_v1.joblib
Tersimpan: outputs/grid_feature_reference.csv
